This notebok walks through how the data is organized by `nsd_data.py`. To use the functions provided here, you will need access to `DeepJuiceDev/juicyfruits/nsd_subset`, which is currently stored on Rusty at `/mnt/ceph/users/alargen/small_nsd/DeepJuiceDev/juicyfruits/nsd_subset`. Navigating through the enitrety of the NSD data stored at `/mnt/ceph/users/gkrawezik/AI_DATASETS/natural-scenes-dataset` is a herculean task that someone else might tackle some day.

# Understanding the Structure of the raw data

## imports + important variables

In [1]:
import pandas as pd
import gdown, tarfile
import os, shutil

In [2]:
voxel_set_name = 'EVC-OTC'
voxel_set = ['EVC','OTC']
image_set = 'shared1000'
path_dir = '/mnt/ceph/users/alargen/small_nsd/DeepJuiceDev/juicyfruits/nsd_subset'
GDRIVE_HEADER = 'https://drive.google.com/uc?export=download&id='

## download the data

if you have the `DeepJuiceDev/juicyfruits/nsd_subset` repo locally, you still need to download certain parts of the response data. this downloads the full `response` folder; the repo doesn't contain the `voxel_beta.csv` files, which contains the voxel responses

helper function from original repo

In [ ]:
def gdrive_download(download_id, down_path, dest_path=None, 
                    extract=True, delete_after=True, **kwargs):
    
    if not download_id.startswith('https://'):
        download_id = GDRIVE_HEADER + download_id
    
    gdown.download(download_id, down_path, **kwargs)
    
    # if download is tarball, extract it
    if 'tar' in down_path and extract:
        tarfile.open(down_path).extractall(dest_path)
    
    if delete_after: # delete the file after download
        if kwargs.get('quiet', False):
            print(f"Deleting download leftovers: {down_path}")
        
        shutil.rmtree(down_path) if os.path.isdir(down_path) else os.remove(down_path)

In [ ]:
drive_id = '1R94PEyTfazeaD0M1YZx4-YJ2NYjQBoFQ'
download_path = f'{path_dir}/response.tar.bz2'

gdrive_download(drive_id, download_path, path_dir,
                extract=True, delete_after=True,)

## loading in the data

the below cell essentially creates all of the necessary paths to the data

In [4]:
stimulus_path = f'{path_dir}/stimulus/{image_set}.csv'
image_root = os.path.join(path_dir, 'stimulus', image_set)

response_path = {}
metadata_path = {}
path_set = [stimulus_path]
for vset in voxel_set:
    response_path[vset] = f'{path_dir}/response/{image_set}_{vset}/voxel_betas.csv'
    metadata_path[vset] = f'{path_dir}/response/{image_set}_{vset}/voxel_metas.csv'
    path_set += [response_path[vset], metadata_path[vset]]

helper functions found in original repo

In [5]:
def load_pandas(path, root=None, **kwargs):
    if root is not None:
        path = os.path.join(root, path)
    
    if not os.path.exists(path):
        raise FileNotFoundError(f"The file does not exist: {path}")

    # Get the file extension
    _, file_extension = os.path.splitext(path)
    file_extension = file_extension.lower()
    
    excel_exts = ['.xls', '.xlsx', '.xlsm', '.xlsb', '.odf', '.ods', '.odt']

    # Match the file extension with the appropriate pandas read function
    if file_extension == '.csv':
        return pd.read_csv(path, **kwargs)
    if file_extension in excel_exts:
        return pd.read_excel(path, **kwargs)
    if file_extension == '.json':
        return pd.read_json(path, **kwargs)
    if file_extension == '.hdf':
        return pd.read_hdf(path, **kwargs)
    if file_extension == '.feather':
        return pd.read_feather(path, **kwargs)
    if file_extension == '.parquet':
        return pd.read_parquet(path, **kwargs)
    if file_extension == '.stata':
        return pd.read_stata(path, **kwargs)
    if file_extension == '.sas':
        return pd.read_sas(path, **kwargs)
    if file_extension == '.pkl':
        return pd.read_pickle(path, **kwargs)
    if file_extension == '.sql':
        # For .sql, a connection is required.
        raise NotImplementedError("Requires a database connection.")
    
    else: # attempt to load with read_table
        try: # takes any filename or buffer
            return pd.read_table(path, **kwargs)
        except Exception as error:
            raise ValueError(f"Failed to read pandas {file_extension}: {error}")

In [6]:
def load_data(data_path):
    if isinstance(data_path, str):
        return load_pandas(data_path).set_index('voxel_id')
    
    if isinstance(data_path, dict):
        response_data = []
        for key, value in data_path.items():
            response_data += [load_pandas(data_path[key])
                                .set_index('voxel_id')]
            
        return pd.concat(response_data)

In [7]:
response_data = load_data(response_path)
metadata = load_data(metadata_path)
stimulus_data = load_pandas(stimulus_path)
n_stimuli = len(stimulus_data)

## looking at the data

rows are voxels, columns are images. value at `(voxel_id, image_id)` gives `voxel_id`'s response to `image_id`.

In [8]:
response_data

,584,605,625,650,1308,1625,1877,2270,2349,2372,...,575701,575971,576011,576749,576789,577817,577964,578169,579906,580813
voxel_id,,,,,,,,,,,,,,,,,,,,,
S1-22-11-34,0.752486,-0.297966,-0.108485,0.027122,0.574349,-0.922022,0.244273,-0.129998,-0.156549,-0.097647,...,0.811811,0.334647,0.576700,-0.051925,0.960291,1.350665,0.745355,-0.587113,-1.248250,-0.684790
S1-22-11-35,-0.103754,0.244961,0.504785,-0.929686,-1.126533,-0.101796,-0.451838,-1.169137,-0.005067,0.310044,...,1.485696,0.213420,1.042587,-0.898903,-0.055290,0.927589,0.634646,0.320112,0.194043,-1.279413
S1-22-12-33,-0.569562,0.082608,0.327680,0.350439,-0.775774,-0.040204,-0.336788,-0.033517,0.059881,0.033386,...,-0.042141,0.052800,0.480060,0.420772,0.358667,0.415686,0.425214,0.281323,-0.422511,-0.626106
S1-23-10-33,0.802062,0.366396,-0.060212,0.717037,-0.578445,0.318957,0.177282,-0.390726,0.566761,0.003886,...,-0.649402,-0.205219,-0.324081,0.658013,0.643523,0.038668,0.335630,-0.877173,-1.218459,-0.321058
S1-23-10-34,0.267148,-0.457386,1.764591,-0.621412,0.296390,-0.758162,-0.369534,-0.403290,-0.816647,-0.263991,...,1.237165,0.443143,1.142742,-0.467000,0.029254,0.263533,0.857254,0.607774,-0.189726,-1.609570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
S7-69-32-38,1.130066,0.006191,0.376372,-0.455574,0.440266,0.575219,-0.120470,-0.248187,-0.456935,0.735229,...,0.499937,-0.639836,-0.054146,-1.270271,-0.360737,-0.090973,0.476141,0.752159,0.516169,0.215909
S7-69-33-38,0.673119,-0.416937,0.267630,-0.422334,0.242033,0.897931,-0.312662,-0.000164,-0.645848,1.020198,...,-0.035872,-0.621331,-0.497352,-1.031812,-0.735381,-0.246403,0.099670,0.556312,-0.195499,0.151886
S7-70-30-35,-0.006916,-0.573220,-0.157642,-0.491851,-0.129971,0.350027,-0.010525,0.686783,-0.322118,-0.335977,...,-0.134072,0.372169,0.267758,-0.763398,-0.043041,0.278724,0.562701,0.707465,-0.865610,0.369399


rows are voxels, columns are different properties of the voxel. for ROI columns, the values are 1 if the voxel is in the ROI and 0 otherwise. important note: some voxels are included more than once as part of different ROIs

In [9]:
metadata.columns

Index(['subj_id', 'ncsnr', 'EVC', 'early', 'ventral', 'midventral', 'lateral',
       'midlateral', 'parietal', 'midparietal', 'V1v', 'V1d', 'V2v', 'V2d',
       'V3v', 'V3d', 'hV4', 'OTC', 'FFA-1', 'FFA-2', 'OFA', 'EBA', 'FBA-1',
       'FBA-2', 'OPA', 'PPA', 'VWFA-1', 'VWFA-2', 'OWFA'],
      dtype='object')

In [11]:
metadata['ncsnr'].describe()

count    44806.000000
mean         0.444016
std          0.204913
min          0.000000
25%          0.290562
50%          0.409630
75%          0.563279
max          1.637091
Name: ncsnr, dtype: float64

provides metadata for the images

In [ ]:
stimulus_data.columns

# Understanding `nsd_data.py`

## imports + important variables

In [12]:
from snap.nsd_data import get_nsd, get_neural_data

In [14]:
data_path = '/mnt/ceph/users/alargen/small_nsd/DeepJuiceDev/juicyfruits/nsd_subset'
region = 'V1'

# all possible region choices:
# functional_rois = ['V1v','V1d','V2v','V2d','V3v','V3d','hV4',
#                   'FFA-1','FFA-2','OFA','EBA','FBA-1','FBA-2',
#                         'OPA','PPA', 'VWFA-1','VWFA-2','OWFA']

# midlevel_rois = {
#     'V1': ['V1v','V1d'],
#     'V2': ['V2v','V2d'],
#     'V3': ['V3v','V3d'],
#     'V4': ['hV4'],
#     'Face Processing': ['FFA-1','FFA-2','OFA'],
#     'Word Processing': ['VWFA-1','VWFA-2','OWFA'],
#     'Body Processing': ['EBA','FBA-1','FBA-2'],
#     'Scene Processing': ['OPA','PPA'], # no RSC?
# }

# global_rois = ['EVC','OTC']
# this code only supports singe regions at a time; can't do a list of regions

subj_subset = [1] # subset of subjects to use; full set: [1, 2, 5, 7]
num_samples = None #1 # number of images to get data from
num_voxels = None #10
dataset = 'response' # can also be 'response_demo' for just the demo data

## loading in the data

`get_nsd` returns dataframes with the data properly filtered.

In [15]:
response_data, stimulus_data, metadata = get_nsd(data_path=data_path, region=region, subj_subset=subj_subset,
                                                 dataset=dataset, num_samples=num_samples, num_voxels=num_voxels)

In [16]:
metadata['ncsnr'].describe()

count    1291.000000
mean        0.566724
std         0.193268
min         0.201034
25%         0.430997
50%         0.565012
75%         0.701304
max         1.199035
Name: ncsnr, dtype: float64

`get_neural_data` loads in the data in a way that `SNAP_analysis_nsd.py` expects. look at `nsd_data.py` for additional input arguments.

In [ ]:
dataloader_neural, images, response = get_neural_data(region=region, data_path=data_path, subj_subset=subj_subset,
                                                      num_samples=num_samples, num_voxels=num_voxels,
                                                      dataset=dataset)